<a href="https://colab.research.google.com/github/r-petrella/Bayesian-Inference/blob/main/final_thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries

In [1]:
%%capture
pip install transformers hdbscan -U plotly==5.19.0 pandas bertopic bitsandbytes


## Upload the corpus

In [2]:
from google.colab import files
uploaded = files.upload()

Saving scopus_20k.csv to scopus_20k.csv


In [3]:
import io
import pandas as pd
data = pd.read_csv(io.BytesIO(uploaded['scopus_20k.csv']))


## Retrieve two interesting columns from data

In [4]:
abstracts = data['Abstract']
titles = data['Title']

## Get the token from Hugging Face to call the LLM

In [5]:
from huggingface_hub import notebook_login
notebook_login()

## Perform 4-bit quantization to lighten the LLM

In [6]:
from torch import cuda

model_id = 'meta-llama/Llama-2-7b-chat-hf'
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

print(device)

cuda:0


In [7]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

## Set the LLM: llama 2 7b

In [8]:
# Llama 2 Tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

# Llama 2 Model
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
#model = model.half().cuda()
model.eval()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Lla

## Parameter for text generation

In [9]:
# Our text generator
generator = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.2,
    max_new_tokens=8000,
    repetition_penalty=1.1
)

## Set the prompts to guide the LLM

In [10]:
# System prompt describes information given to all conversations
system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful and honest assistant for labeling topics.
<</SYS>>
"""

In [11]:
# Example prompt demonstrating the output we are looking for
example_prompt = """
I have a topic that contains the following documents:
- Psychological readiness following anterior cruciate ligament reconstruction (ACLR) correlates with different return to sport outcomes. However, the relationship between strength and power and psychological readiness remains unexplored. The aim of this study was to investigate the relationship between anterior cruciate ligament return to sport after injury (ACL-RSI) score...
- Background: Achilles tendon rupture is common among physically active individuals, yet a high percentage fail to return to their former activity after the injury. Quantifiable factors such as type of treatment, hours of rehabilitation, and age have not been associated with return-to-play rates. A factor that influences recovery is the participant’s experience before and throughout the rehabilitation process...
- The return to field is a critical moment for an athlete who has dislocated his shoulder as there is a significant risk of recurrence. The decision to return to field made by the doctor will therefore be crucial for the smooth continuation of the athlete's career. Hypothesis: This objective is to compare the criteria most used by specialists in clearing an overhead athlete to return to competition after a first episode of antero-internal dislocation of the glenohumeral joint with or without surgery...

The topic is described by the following keywords: 'return, to, the, criteria, athlete, and, by, of, quadriceps, achilles'.

Based on the information about the topic above, please create a very short label of this topic. Make sure you to only return the label and nothing more.

[/INST] Return to Sports Criteria After Injury
"""

In [12]:
# Our main prompt with documents ([DOCUMENTS]) and keywords ([KEYWORDS]) tags
main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
[/INST]
"""

In [13]:
prompt = system_prompt + example_prompt + main_prompt

## Embedding

In [14]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)
embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

array([[-0.04074085, -0.02305938,  0.00682972, ..., -0.05615811,
         0.05217776, -0.01082609],
       [-0.01773744, -0.00678086,  0.02960893, ..., -0.08243284,
         0.02831526,  0.00138619],
       [-0.03586454, -0.02530961,  0.02732268, ..., -0.05108161,
         0.05178559,  0.02192785],
       ...,
       [-0.06335061, -0.01502493,  0.01723955, ..., -0.03623749,
         0.01387563,  0.00147533],
       [-0.01094195, -0.03268239,  0.04085312, ..., -0.08055165,
         0.04507776,  0.03885034],
       [-0.01995889, -0.0178422 ,  0.01809282, ..., -0.04787526,
         0.02216915,  0.03348066]], dtype=float32)

In [15]:
type(embeddings)
print(embeddings.shape)

(20000, 384)


## Clustering

In [16]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=200, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [17]:
# Pre-reduce embeddings for visualization purposes
reduced_embeddings = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)

/usr/local/lib/python3.10/dist-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


## Fine Tuning of Topics

In [18]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

# Text generation with Llama 2
llama2 = TextGeneration(generator, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    "Llama2": llama2,
    "MMR": mmr,
}

## Unsupervised Training

In [20]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Sub-models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings)

2024-08-04 10:00:35,663 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-08-04 10:00:57,520 - BERTopic - Dimensionality - Completed ✓
2024-08-04 10:00:57,522 - BERTopic - Cluster - Start clustering the reduced embeddings
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()
2024-08-04 10:01:00,176 - BERTopic - Cluster - Completed ✓
2024-08-04 10:01:00,185 - BERTopic - Representation - Extracting topics from clusters using representation models.
100%|██████████| 11/11 [03:07<00:00, 17.08s/it]
2024-08-04 10:04:14,689 - BERTopic - Representation - Completed ✓


## Results

In [21]:
# Show topics
info = topic_model.get_topic_info()
info

,Topic,Count,Name,Representation,KeyBERT,Llama2,MMR,Representative_Docs
0,-1,5427,-1_the_and_of_in,"[the, and, of, in, to, with, were, was, for, on]","[knee, injury, exercise, patients, interventio...","[Return of Sports After Injury, , , , , , , , , ]","[the, of, with, were, group, patients, results...",[Purpose: A treatment-specific rehabilitation ...
1,0,4408,0_and_the_of_in,"[and, the, of, in, to, with, exercise, physica...","[exercise, fitness, health, patients, interven...",[Exercise and Physical Activity in Older Adult...,"[of, exercise, physical, health, training, int...",[Background The use of mobile health applicati...
2,1,3250,1_the_and_of_in,"[the, and, of, in, to, were, for, was, with, on]","[sport, athletes, sports, injury, injuries, so...","[Return to Play After Injury, , , , , , , , , ]","[were, players, training, athletes, injury, du...",[Purpose: The purpose of this systematic revie...
3,2,1573,2_the_and_of_acl,"[the, and, of, acl, to, knee, in, aclr, with, ...","[acl, aclr, meniscal, ligament, injury, anteri...","[Return to Sports After ACL Injury, , , , , , ...","[acl, knee, aclr, patients, ligament, at, cruc...",[Background: Reconstruction using autograft re...
4,3,1088,3_the_and_of_shoulder,"[the, and, of, shoulder, patients, to, in, wit...","[arthroscopic, shoulder, tendon, postoperative...","[Rotator Cuff Repair Outcomes, , , , , , , , , ]","[of, shoulder, patients, rotator, arthroscopic...",[Objective. The objective of this retrospectiv...
5,4,990,4_pain_the_and_of,"[pain, the, and, of, to, in, with, back, for, ...","[physiotherapy, physiotherapists, musculoskele...",[Pain and Disability in Musculoskeletal Condit...,"[pain, of, lbp, low, patients, chronic, disabi...",[Introduction and objectives: Chronic low back...
6,5,887,5_and_the_of_hip,"[and, the, of, hip, to, in, patients, with, kn...","[osteoarthritis, arthroplasty, hip, arthroscop...","[Hip and Knee Osteoarthritis Treatment, , , , ...","[hip, patients, knee, osteoarthritis, oa, pain...",[Background: Although the initial treatment re...
7,6,805,6_concussion_and_of_to,"[concussion, and, of, to, the, in, with, injur...","[concussion, concussions, injury, injuries, sp...","[Concussion and Brain Injury, , , , , , , , , ]","[concussion, of, injury, mtbi, brain, symptom,...",[Advanced magnetic resonance imaging (MRI) tec...
8,7,558,7_cancer_and_the_of,"[cancer, and, the, of, to, exercise, in, patie...","[prehabilitation, oncology, exercise, cancer, ...","[Exercise Interventions for Cancer Patients, ,...","[cancer, of, exercise, patients, physical, sur...",[Background: Patients with breast cancer under...
9,8,528,8_covid_19_the_and,"[covid, 19, the, and, of, in, to, pandemic, wi...","[covid, coronavirus, patients, cov, pandemic, ...","[Return to Sports After COVID-19, , , , , , , ...","[covid, pandemic, were, physical, health, pati...",[Purpose: Covid-19 is a viral infection that a...


In [22]:
info.iloc[:,6]  # keywords from MMR

,MMR
0,"[the, of, with, were, group, patients, results, pain, exercise, muscle]"
1,"[of, exercise, physical, health, training, intervention, results, pa, older, adults]"
2,"[were, players, training, athletes, injury, during, ankle, strength, results, muscle]"
3,"[acl, knee, aclr, patients, ligament, at, cruciate, injury, graft, surgery]"
4,"[of, shoulder, patients, rotator, arthroscopic, results, outcomes, after, clinical, surgery]"
5,"[pain, of, lbp, low, patients, chronic, disability, exercise, results, care]"
6,"[hip, patients, knee, osteoarthritis, oa, pain, tka, after, surgery, results]"
7,"[concussion, of, injury, mtbi, brain, symptom, tbi, athletes, src, related]"
8,"[cancer, of, exercise, patients, physical, survivors, breast, intervention, treatment, pa]"
9,"[covid, pandemic, were, physical, health, patients, infection, after, exercise, rehabilitation]"


Changing labels to llama2

In [23]:
llama2_labels = [label[0][0].split("\n")[0] for label in topic_model.get_topics(full=True)["Llama2"].values()]
topic_model.set_topic_labels(llama2_labels)


llama2_labels

['Return of Sports After Injury',
 'Exercise and Physical Activity in Older Adults',
 'Return to Play After Injury',
 'Return to Sports After ACL Injury',
 'Rotator Cuff Repair Outcomes',
 'Pain and Disability in Musculoskeletal Conditions',
 'Hip and Knee Osteoarthritis Treatment',
 'Concussion and Brain Injury',
 'Exercise Interventions for Cancer Patients',
 'Return to Sports After COVID-19',
 'Return of Sports Criteria']

In [ ]:
topic_model.get_topic(4, full=True)["KeyBERT"]

[('physiotherapy', 0.8527564),
 ('physiotherapists', 0.85252106),
 ('musculoskeletal', 0.85215473),
 ('pain', 0.8429741),
 ('spine', 0.8393389),
 ('lumbar', 0.8380296),
 ('patients', 0.82184434),
 ('muscle', 0.8131739),
 ('patient', 0.81308323),
 ('neck', 0.8115961)]

In [36]:
topic_model.topic_representations_

{-1: [('the', 0.05367179705668745),
  ('and', 0.04821396841676325),
  ('of', 0.044588989614279526),
  ('in', 0.036045808222996426),
  ('to', 0.03476329597146659),
  ('with', 0.02630094269021466),
  ('were', 0.02417932309669333),
  ('was', 0.02310760450283329),
  ('for', 0.022721374468014168),
  ('on', 0.01629771623065858)],
 0: [('and', 0.04969056341443206),
  ('the', 0.04787693171563075),
  ('of', 0.04233327857408645),
  ('in', 0.038136498234025526),
  ('to', 0.03479425401608793),
  ('with', 0.026876146340233255),
  ('exercise', 0.022911961538092575),
  ('physical', 0.02276011363447327),
  ('for', 0.02217114535150481),
  ('were', 0.022127059901136253)],
 1: [('the', 0.057164613102868495),
  ('and', 0.04869390379683837),
  ('of', 0.04263601749148113),
  ('in', 0.03630811129342924),
  ('to', 0.03430866655500408),
  ('were', 0.022885077014513155),
  ('for', 0.022417014966140278),
  ('was', 0.021213206490599178),
  ('with', 0.0210473798695509),
  ('on', 0.017623029983254736)],
 2: [('the'

## LDAvis

In [ ]:
topic_model.visualize_topics(custom_labels=llama2_labels)

## UMAP with topics and Titles

In [ ]:
topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings,
                                custom_labels=llama2_labels, hide_annotations=True)

In [ ]:
topic_model.get_topic(topic=7)

[('achilles', 0.03831262579415172),
 ('tendon', 0.030114277706387884),
 ('rupture', 0.01371182292481602),
 ('tendinopathy', 0.0124526150680075),
 ('atr', 0.00842654477872047),
 ('repair', 0.008300233216973986),
 ('ruptures', 0.007727749397429049),
 ('visaa', 0.007334674888567459),
 ('treatment', 0.006419594932535029),
 ('atrs', 0.006382119708234023)]

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
info_docs = topic_model.get_document_info(abstracts)
info_docs
# Save the combined top keywords DataFrame to a CSV file
info_docs.to_csv(r'/content/drive/My Drive/llama2_20k_clusters.csv',
                        index=False )

In [ ]:
topic_model.get_representative_docs(topic=7)

['Purpose: Evaluate the one-year postoperative outcomes in patients with Chronic Achilles tendon rupture. Methods: Patients surgically treated for Chronic Achilles tendon rupture (n = 22, 14 males and 8 females, mean age 61 ± 15) were evaluated by Achilles tendon Total Rupture Score, The Physical Activity Scale, The Foot and Ankle Outcome Score, Calf muscle endurance test, counter movement jump, Hopping, ultrasound measurement of tendon length, Achilles Tendon Resting Angle, dorsi flexion range of motion and calf muscle circumference. Muscle function and tendon length outcomes on the injured side were compared with the healthy side. Results: The patients scored a mean of 62 ± 26 on the Achilles tendon Total Rupture Score. Median scores on the injured compared with the healthy side were lower in heel-rise repetitions (20 vs 24\xa0cm, p = 0.004), hel-rise height (8 vs 10\xa0cm, p < 0.001), heel-rise total work (872 vs 1590\xa0joule, p < 0.001) and hopping ratio (0.37 vs 0.48, p = 0.005).

In [ ]:
reduced_model = topic_model.reduce_topics(abstracts, nr_topics=10)
reduced_model

In [ ]:
# Show topics
info2 = reduced_model.get_topic_info()
info2

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6968,-1_the_and_of_in,"[the, and, of, in, to, with, were, for, was, on]",[Background: The chronic pain of patients with...
1,0,9821,0_the_and_of_in,"[the, and, of, in, to, with, were, for, was, on]",[Korea already entered the aging society (Augu...
2,1,2701,1_the_and_of_to,"[the, and, of, to, in, patients, with, were, w...",[The ACL is the primary stabilizer of the knee...
3,2,238,2_the_and_of_in,"[the, and, of, in, to, tendon, that, expressio...",[Background: Various muscle contraction modali...
4,3,129,3_abstract_available_no_,"[abstract, available, no, , , , , , , ]","[[No abstract available], [No abstract availab..."
5,4,50,4_the_machine_of_and,"[the, machine, of, and, learning, model, to, f...",[Background: Given the intricate and grave nat...
6,5,36,5_causal_the_and_genetic,"[causal, the, and, genetic, of, polymorphisms,...",[Purpose: This study aims to assess the causal...
7,6,31,6_microbiota_gut_the_and,"[microbiota, gut, the, and, of, in, intestinal...",[Background: Exercise can modulate gut microbi...
8,7,15,7_sensing_the_sensor_sensors,"[sensing, the, sensor, sensors, and, to, of, m...",[Gait analysis refers to the systematic study ...
9,8,11,8_cannabis_cbd_use_of,"[cannabis, cbd, use, of, and, the, exercise, i...",[Background: Ankle fractures are common orthop...


## Semi-Supervised Training

In [57]:
import numpy as np
seed_topic_list = [
    ['mental', 'health', 'psychological', 'anxiety'],
    ["surgery","repair","postoperative","clinical"],
    ["cell", "tissue", "cartilage", "human"]
    ]

# Check lengths of all sublists
lengths = [len(sublist) for sublist in seed_topic_list]
assert len(set(lengths)) == 1, "Sublists have different lengths"

# Verify all elements are strings
for sublist in seed_topic_list:
    for item in sublist:
        assert isinstance(item, str), f"Non-string element found: {item}"

# Convert to NumPy array
try:
    array = np.array(seed_topic_list)
    print("NumPy array created successfully:")
    print(array)
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Sub-models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,
  seed_topic_list=seed_topic_list,
  # Hyperparameters
  top_n_words=10,
  verbose=True
)




# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings)

2024-08-03 17:57:12,600 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [ ]:
info = topic_model.get_topic_info()
info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7033,-1_the_and_of_in,"[the, and, of, in, study, to, for, were, was, ...",[Objective: To evaluate hip and knee muscular ...
1,0,802,0_concussion_mtbi_brain_tbi,"[concussion, mtbi, brain, tbi, symptom, sympto...",[Objective:To determine the number of prior co...
2,1,544,1_cancer_breast_survivors_exercise,"[cancer, breast, survivors, exercise, chemothe...",[Background: An increased number of breast can...
3,2,533,2_covid19_pandemic_infection_postcovid19,"[covid19, pandemic, infection, postcovid19, sa...",[Background: Students were at an increased ris...
4,3,355,3_pain_lbp_back_low,"[pain, lbp, back, low, care, disability, chron...",[Background: Clinical practice guidelines prom...
...,...,...,...,...,...
281,280,10,280_meniscus_nondiscoid_repair_meniscal,"[meniscus, nondiscoid, repair, meniscal, disco...",[Purpose: This study aimed to investigate clin...
282,281,10,281_running_runners_stride_speed,"[running, runners, stride, speed, attenuation,...",[Background: An increasing number of commercia...
283,282,10,282_hypermobility_wrist_cwh_joint,"[hypermobility, wrist, cwh, joint, hand, jh, g...",[Background: Joint hypermobility is a spectrum...
284,283,10,283_sitting_inactivity_pa_sbq,"[sitting, inactivity, pa, sbq, energydense, mi...",[Background: Physical inactivity and obesity c...


## Hierarchical clustering based on the c-TF-IDF matrix

In [40]:
from scipy.cluster import hierarchy as sch
from bertopic import BERTopic
topic_model2 = BERTopic()
topics2, probs2 = topic_model.fit_transform(abstracts)

# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'ward', optimal_ordering=True)
hierarchical_topics = topic_model.hierarchical_topics(titles, linkage_function=linkage_function)

hierarchical_topics[["Topics","Distance"]]

,Topics,Distance
8,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",0.424580
7,"[0, 1, 4, 6, 7, 8]",0.342652
6,"[0, 1, 4, 6, 7]",0.316966
5,"[2, 3, 5, 9]",0.297302
4,"[0, 1, 4, 7]",0.254964
3,"[2, 3, 5]",0.204245
2,"[0, 1, 4]",0.184815
1,"[3, 5]",0.179349
0,"[0, 1]",0.125255


In [35]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, custom_labels=llama2_labels)


In [ ]:
#@title Convert ipynb to HTML in Colab
# Upload ipynb
from google.colab import files
f = files.upload()

# Convert ipynb to html
import subprocess
file0 = list(f.keys())[0]
_ = subprocess.run(["pip", "install", "nbconvert"])
_ = subprocess.run(["jupyter", "nbconvert", file0, "--to", "html"])

# download the html
files.download(file0[:-5]+"html")

@article{grootendorst2022bertopic,
  title={BERTopic: Neural topic modeling with a class-based TF-IDF procedure},
  author={Grootendorst, Maarten},
  journal={arXiv preprint arXiv:2203.05794},
  year={2022}
}
